# Smart Logistics and Supply Chain Analytics using Apache Spark


# Q1. Spark Initialization and Data Loading

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark=SparkSession.builder.appName("Logistics").getOrCreate()

In [2]:
# Load logistics datasets.

shipments=spark.read.csv("/content/shipments.csv",header=True,inferSchema=True)
warehouse=spark.read.csv("/content/warehouse.csv",header=True,inferSchema=True)
gps=spark.read.csv("/content/gps_tracking.csv",header=True,inferSchema=True)
fuel=spark.read.csv("/content/fuel.csv",header=True,inferSchema=True)
customers=spark.read.csv("/content/customers.csv",header=True,inferSchema=True)

for df in [shipments,warehouse,gps,fuel,customers]:
    df.printSchema()
    df.show(5)


root
 |-- shipment_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- expected_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- weight_kg: double (nullable = true)
 |-- distance_km: double (nullable = true)

+-----------+-------+-----------+-----------+----------+----------------------+--------------------+----------+---------+-----------+
|shipment_id| region|     origin|destination| ship_date|expected_delivery_date|actual_delivery_date|    status|weight_kg|distance_km|
+-----------+-------+-----------+-----------+----------+----------------------+--------------------+----------+---------+-----------+
|   SHP00001|  North|      Delhi| Chandigarh|2026-03-04|            2026-03-06|          2026-03-06| Delivered|   446.63|     176.06|
|   SHP00002|   West|     Mumbai|

## Q2 RDD Implementation

In [14]:
# Transformation
rdd=shipments.rdd
print(rdd.count())
print(rdd.first())
print(rdd.take(3))

# Action
rdd.map(lambda x:x[1]).distinct().collect()
rdd.filter(lambda x:x.status=="Delayed").count()


2000
Row(shipment_id='SHP00001', region='North', origin='Delhi', destination='Chandigarh', ship_date=datetime.date(2026, 3, 4), expected_delivery_date=datetime.date(2026, 3, 6), actual_delivery_date=datetime.date(2026, 3, 6), status='Delivered', weight_kg=446.63, distance_km=176.06)
[Row(shipment_id='SHP00001', region='North', origin='Delhi', destination='Chandigarh', ship_date=datetime.date(2026, 3, 4), expected_delivery_date=datetime.date(2026, 3, 6), actual_delivery_date=datetime.date(2026, 3, 6), status='Delivered', weight_kg=446.63, distance_km=176.06), Row(shipment_id='SHP00002', region='West', origin='Mumbai', destination='Ahmedabad', ship_date=datetime.date(2026, 1, 24), expected_delivery_date=datetime.date(2026, 1, 26), actual_delivery_date=None, status='In-Transit', weight_kg=103.42, distance_km=992.33), Row(shipment_id='SHP00003', region='Central', origin='Nagpur', destination='Bhopal', ship_date=datetime.date(2026, 4, 25), expected_delivery_date=datetime.date(2026, 4, 30), 

815

## Q3. Key-Value Operations and Persistence

In [4]:
pair=rdd.map(lambda x:(x.region,1))
regionCount=pair.reduceByKey(lambda a,b:a+b).cache()
regionCount.collect()
regionCount.sortByKey().collect()


[('Central', 412),
 ('East', 417),
 ('North', 393),
 ('South', 372),
 ('West', 406)]

## Q4. Spark DataFrame Operations

In [5]:
shipments.filter(col("status")=="Delayed").show()
shipments.groupBy("region").agg(count("*").alias("Total")).show()
shipments.join(customers,"shipment_id").select("shipment_id","region","rating").show()


+-----------+-------+-----------+-----------+----------+----------------------+--------------------+-------+---------+-----------+
|shipment_id| region|     origin|destination| ship_date|expected_delivery_date|actual_delivery_date| status|weight_kg|distance_km|
+-----------+-------+-----------+-----------+----------+----------------------+--------------------+-------+---------+-----------+
|   SHP00006|   East|      Patna|    Kolkata|2026-01-18|            2026-01-19|          2026-01-22|Delayed|     44.5|     387.55|
|   SHP00007|  North| Chandigarh|     Jaipur|2026-04-27|            2026-05-03|          2026-05-08|Delayed|   108.71|     437.12|
|   SHP00009|Central|     Bhopal|     Nagpur|2026-01-15|            2026-01-17|          2026-01-22|Delayed|   137.53|     355.93|
|   SHP00011|  South|  Hyderabad|  Bangalore|2026-05-30|            2026-06-03|          2026-06-05|Delayed|   498.78|     788.81|
|   SHP00015|Central|     Bhopal|     Raipur|2026-03-18|            2026-03-25|    

## Q5 Exploratory Data Analysis and Spark SQL

In [19]:
shipments.createOrReplaceTempView("shipments")
warehouse.createOrReplaceTempView("warehouse")
fuel.createOrReplaceTempView("fuel")

# spark.sql("select region,count(*) Total from shipments group by region").show()
# spark.sql("select status,count(*) from shipments group by status").show()
# spark.sql("select region,avg(weight_kg) AvgWeight from shipments group by region").show()


In [21]:
from pyspark.sql.functions import count

shipments.groupBy("region") \
         .agg(count("*").alias("Total Shipments")) \
         .show()

+-------+---------------+
| region|Total Shipments|
+-------+---------------+
|  South|            372|
|Central|            412|
|   East|            417|
|   West|            406|
|  North|            393|
+-------+---------------+



In [22]:
shipments.groupBy("status") \
         .count() \
         .show()

+----------+-----+
|    status|count|
+----------+-----+
|In-Transit|   86|
| Cancelled|   64|
| Delivered| 1035|
|   Delayed|  815|
+----------+-----+



In [23]:
from pyspark.sql.functions import avg

shipments.groupBy("region") \
         .agg(avg("weight_kg").alias("Average Weight")) \
         .show()

+-------+------------------+
| region|    Average Weight|
+-------+------------------+
|  South|248.85833333333335|
|Central|257.68662621359204|
|   East|243.10937649880103|
|   West| 258.3577586206896|
|  North|  251.321475826972|
+-------+------------------+



## Q6 ETL Pipeline Development

In [24]:
from pyspark.sql.functions import *

# Extract
clean = shipments

# Transform
clean = clean.dropDuplicates()
clean = clean.na.drop()
clean = clean.withColumn("ship_date", to_date("ship_date"))
clean = clean.withColumn("actual_delivery_date", to_date("actual_delivery_date"))
clean = clean.withColumnRenamed("weight_kg", "weight")

# Load
clean.write.mode("overwrite").csv("clean_shipments", header=True)

# Verify
clean.show(5)

+-----------+-------+----------+-----------+----------+----------------------+--------------------+---------+------+-----------+
|shipment_id| region|    origin|destination| ship_date|expected_delivery_date|actual_delivery_date|   status|weight|distance_km|
+-----------+-------+----------+-----------+----------+----------------------+--------------------+---------+------+-----------+
|   SHP00091|   East|   Kolkata|Bhubaneswar|2026-05-12|            2026-05-19|          2026-05-19|Delivered|209.56|     321.99|
|   SHP00135|   West|      Pune|  Ahmedabad|2026-02-10|            2026-02-13|          2026-02-13|Delivered|285.96|     173.19|
|   SHP00239|Central|    Nagpur|     Raipur|2026-04-15|            2026-04-17|          2026-04-17|Delivered|239.39|    1281.54|
|   SHP00959|   East|   Kolkata|Bhubaneswar|2026-01-12|            2026-01-19|          2026-01-22|  Delayed|305.35|     630.24|
|   SHP01398|  North|Chandigarh|     Jaipur|2026-05-01|            2026-05-04|          2026-05-0

## Q7. Machine Learning/Deep Learning Implementation

In [8]:
from pyspark.ml.feature import StringIndexer,VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator


In [9]:
indexer=StringIndexer(inputCol="status",outputCol="label")
assembler=VectorAssembler(inputCols=["weight_kg","distance_km"],outputCol="features")
dt=DecisionTreeClassifier(labelCol="label",featuresCol="features")
pipeline=Pipeline(stages=[indexer,assembler,dt])


In [10]:
train,test=shipments.randomSplit([0.8,0.2],42)

In [11]:
model=pipeline.fit(train)
predictions=model.transform(test)

predictions.select("status","prediction").show(10)

+----------+----------+
|    status|prediction|
+----------+----------+
|In-Transit|       0.0|
|   Delayed|       0.0|
|   Delayed|       0.0|
| Cancelled|       1.0|
|In-Transit|       0.0|
| Delivered|       0.0|
| Delivered|       1.0|
| Delivered|       1.0|
| Delivered|       1.0|
|   Delayed|       0.0|
+----------+----------+
only showing top 10 rows


In [12]:
accuracy_eval=MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
f1_eval=MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

accuracy=accuracy_eval.evaluate(predictions)
f1=f1_eval.evaluate(predictions)

print("Accuracy :",accuracy)
print("F1 Score :",f1)

Accuracy : 0.5027932960893855
F1 Score : 0.45698755362834304
